## CarPrice Model

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


# -------------------- Load Data --------------------
CarPrice = pd.read_csv("CarPrice_Assignment.csv")

# Dropping unwanted columns
x = CarPrice.drop(['price', 'CarName', 'car_ID', 'symboling', 'stroke'], axis=1)
y = CarPrice.price

# Train/Test Split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.2, random_state=121)


# -------------------- Pipelines --------------------
num_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('Scale', StandardScaler())
])

cat_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

cat_pipeline2 = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('OneHotEncoder', OneHotEncoder())
])


# Columns
num_col = x.select_dtypes(include=['int64', 'float64']).columns
cat_col = ['aspiration', 'fuelsystem']
cat_col2 = ['fueltype', 'doornumber', 'carbody',
            'drivewheel', 'enginelocation', 'enginetype', 'cylindernumber']


# Final Column Transformer
final_pipeline = ColumnTransformer([
    ('num_pipeline', num_pipeline, num_col),
    ('cat_pipeline', cat_pipeline, cat_col),
    ('cat_pipeline2', cat_pipeline2, cat_col2)
])

# Fit Pipeline
final_pipeline.fit(x_train)

# Transform
X_train_tr = final_pipeline.transform(x_train)
X_test_tr = final_pipeline.transform(x_test)


# -------------------- Train Model --------------------
Model = RandomForestRegressor()
Model.fit(X_train_tr, y_train)

y_train_pred = Model.predict(X_train_tr)
y_test_pred = Model.predict(X_test_tr)



def predict_user_input():

    user_data = {}

    print("\n======= Enter Car Features for Price Prediction =======\n")

    # Numeric features (show unique sorted values)
    for col in num_col:
        options = sorted(CarPrice[col].unique())
        print(f"\nEnter value for {col}")
        print(f"Your options are: {tuple(options[:10])} ...")  # showing first 10 to avoid long lists
        value = float(input("Enter value: "))
        user_data[col] = value

    # Ordinal categorical features
    for col in cat_col:
        options = list(CarPrice[col].unique())
        print(f"\nEnter value for {col}")
        print(f"Your options are: {tuple(options)}")
        value = input("Enter value: ")
        user_data[col] = value

    # One-hot categorical features
    for col in cat_col2:
        options = list(CarPrice[col].unique())
        print(f"\nEnter value for {col}")
        print(f"Your options are: {tuple(options)}")
        value = input("Enter value: ")
        user_data[col] = value

    # Convert to DataFrame
    user_df = pd.DataFrame([user_data])

    # Transform input
    transformed = final_pipeline.transform(user_df)

    # Predict
    predicted_price = Model.predict(transformed)

    print("\n.........................................")
    print(f"Predicted Car Price = ${predicted_price[0]:,.2f}")
    print("...........................................\n")



# -------------------- Results --------------------
print(f"Training_error: {mean_squared_error(y_train, y_train_pred):.2f}")
print(f"Training_Accuracy: {r2_score(y_train, y_train_pred):.2f}")
print()
print(f"Testing_error: {mean_squared_error(y_test, y_test_pred):.2f}")
print(f"Testing_Accuracy: {r2_score(y_test, y_test_pred):.2f}")

# Call user input prediction
predict_user_input()

Training_error: 720156.77
Training_Accuracy: 0.99

Testing_error: 3404280.84
Testing_Accuracy: 0.94

======= Enter Car Features for Price Prediction =======


Enter value for wheelbase
Your options are: (np.float64(86.6), np.float64(88.4), np.float64(88.6), np.float64(89.5), np.float64(91.3), np.float64(93.0), np.float64(93.1), np.float64(93.3), np.float64(93.7), np.float64(94.3)) ...


KeyboardInterrupt: Interrupted by user